# VisionGuard AI - Inference & Explainability Demo
This notebook demonstrates model inference, Grad-CAM generation, and ONNX Runtime performance for U-Net.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from src.config import load_config
from src.models.unet import UNet
from src.data.dataset import MedicalSegmentationDataset
from src.utils.explainability import SegmentationGradCAM, overlay_heatmap

## 1. Load Dataset and Model

In [ ]:
config = load_config('../configs/config.yaml')
dataset = MedicalSegmentationDataset(generate_synthetic=True, num_synthetic_samples=10, image_size=256)
sample = dataset[0]
image_tensor = sample['image'].unsqueeze(0)
mask_tensor = sample['mask']

model = UNet(in_channels=config.unet_in_channels, num_classes=config.unet_num_classes)
model.eval()
print('Model loaded successfully.')

## 2. Grad-CAM Explainability Map

In [ ]:
target_layer = model.up4.conv.double_conv[3]
grad_cam = SegmentationGradCAM(model, target_layer)

with torch.enable_grad():
    image_tensor.requires_grad = True
    heatmap = grad_cam.generate_cam(image_tensor, class_idx=0)
grad_cam.remove_hooks()

# Denormalize image for plotting
orig_img = image_tensor[0].detach().numpy().transpose(1, 2, 0)
orig_img = orig_img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
orig_img = np.clip(orig_img, 0, 1)

blended = overlay_heatmap(orig_img, heatmap, alpha=0.4)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig_img)
axes[0].set_title('Original Scan')
axes[0].axis('off')

axes[1].imshow(mask_tensor[0].numpy(), cmap='gray')
axes[1].set_title('Target Mask')
axes[1].axis('off')

axes[2].imshow(blended)
axes[2].set_title('Grad-CAM Explanation')
axes[2].axis('off')

plt.show()